# HCE v9 — Stage 1: Inference-only fixes on v8 checkpoint

**No retraining.** Reuses `multi_tissue_v8_results/hce_v8_best_model.pt`.

Stage 1 changes vs v8:
1. **S1-A**: Mask `(unspecified)` leaves from leaf argmax — forces commitment to specific subtypes
2. **S1-B**: Real Brain_normal protocol diagnostic (via total counts) + flip `snrna_seq` to `False`
3. **S1-C**: Adaptive ancestor roll-up — prevents rolling up to overly-broad parents like `leukocyte`
4. **S1-D**: Diagnostic confirming which v8 `TARGETED_WEIGHT_BOOSTS` entries silently no-op'd after `deduplicate_hierarchy()`

Outputs go to `multi_tissue_v9_results/`. If Stage 1 acceptance fails, Stage 2 retrain is in the plan file.


## 1. Imports

In [ ]:
import os, sys, warnings, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import Counter

import scanpy as sc
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import scipy.sparse as sp
import h5py

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), 'src'))
from cell2sentence.hce_trainer import build_reachability_matrix_from_ontology
from cell2sentence.hierarchy_utils import deduplicate_hierarchy

warnings.filterwarnings('ignore')
print('=' * 60)
print('  IMPORTS OK')
print(f'  PyTorch : {torch.__version__}')
print(f'  CUDA    : {torch.cuda.is_available()} | devices: {torch.cuda.device_count()}')
print('=' * 60)

## 2. Configuration

In [ ]:
# ── Output ────────────────────────────────────────────────────────────────────
# v9 Stage 1 — write new outputs to v9 dir but READ v8 checkpoint + temperature
OUT_DIR          = 'multi_tissue_v9_results'
V8_RESULTS_DIR   = 'multi_tissue_v8_results'
HCE_MODEL_PATH   = os.path.join(V8_RESULTS_DIR, 'hce_v8_best_model.pt')   # reused, not overwritten
TEMPERATURE_PATH = os.path.join(V8_RESULTS_DIR, 'temperature_v8.pt')      # reused, not overwritten
C2S_MODEL_NAME   = 'vandijklab/C2S-Pythia-410m-cell-type-conditioned-cell-generation'

# ── ORGAN_CONFIGS (training data; 6 organs, ids 0–5) ─────────────────────────
# 'max_cells'  — per-organ-per-leaf cell cap
# 'snrna_seq'  — when True, cell sentences are ranked by expression/gene_length
ORGAN_CONFIGS = [
    {'name': 'lung',         'path': 'lung.h5ad',                       'label_col': 'ann_finest_level', 'gene_col': 'feature_name', 'hierarchy_cols': ['ann_level_1','ann_level_2','ann_level_3','ann_level_4','ann_level_5'], 'coarse_col': None,                 'id': 0, 'max_cells':  300, 'snrna_seq': False},
    {'name': 'brain_glia',   'path': 'brain_new.h5ad',                  'label_col': 'cell_type',        'gene_col': 'feature_name', 'hierarchy_cols': [],                                                                  'coarse_col': 'supercluster_term',  'id': 1, 'max_cells':  900, 'snrna_seq': True},
    {'name': 'brain_neurons','path': 'brain_neurons_processed.h5ad',    'label_col': 'label',            'gene_col': 'feature_name', 'hierarchy_cols': [],                                                                  'coarse_col': None,                 'id': 2, 'max_cells':  900, 'snrna_seq': True},
    {'name': 'liver',        'path': 'census_data/liver.h5ad',          'label_col': 'cell_type',        'gene_col': 'feature_name', 'hierarchy_cols': [],                                                                  'coarse_col': None,                 'id': 3, 'max_cells':  300, 'snrna_seq': False},
    {'name': 'lymph_node',   'path': 'census_data/lymph_node.h5ad',     'label_col': 'cell_type',        'gene_col': 'feature_name', 'hierarchy_cols': [],                                                                  'coarse_col': None,                 'id': 4, 'max_cells':  300, 'snrna_seq': False},
    {'name': 'bone_marrow',  'path': 'census_data/bone_marrow.h5ad',    'label_col': 'cell_type',        'gene_col': 'feature_name', 'hierarchy_cols': [],                                                                  'coarse_col': None,                 'id': 5, 'max_cells':  500, 'snrna_seq': False},
]

# ── PER_LEAF_MAX_BY_ORGAN: extra cap on specific (organ, leaf-label) combos ──
# Used by load_organ() AFTER the cfg['max_cells'] sampler. Mitigates the v7
# "Peribronchial fibroblasts" attractor by reducing lung fibroblast-subtype mass.
PER_LEAF_MAX_BY_ORGAN = {
    'lung': {
        'Peribronchial fibroblasts':     100,
        'Adventitial fibroblasts':       100,
        'Alveolar fibroblasts':          100,
        'Subpleural fibroblasts':        100,
        'Myofibroblasts':                100,
        'Smooth muscle FAM83D+':         100,
        'SM activated stress response':  100,
    },
}

# ── LAB_CONFIGS (zero-shot eval). snrna_seq controls inference-time length norm ─
LAB_CONFIGS = [
    {'name': 'All_cells (glioma)', 'path': 'All_cells.h5ad',                  'label_col': 'predicted.high_hierarchy', 'snrna_seq': False},
    {'name': 'Brain_normal',       'path': 'lab-data/Brain_normal.h5ad',      'label_col': 'cell_type',                'snrna_seq': False},  # v9 Stage 1: flipped from True; S1-B diagnostic confirms protocol
    {'name': 'Liver_normal',       'path': 'lab-data/Liver_normal.h5ad',      'label_col': 'cell_type',                'snrna_seq': False},
    {'name': 'Lymph_node_normal',  'path': 'lab-data/Lymph_node_normal.h5ad', 'label_col': 'cell_type',                'snrna_seq': False},
    {'name': 'lymphoid',           'path': 'lab-data/lymphoid.h5ad',          'label_col': 'cell_type',                'snrna_seq': False},
    {'name': 'myeloid',            'path': 'lab-data/myeloid.h5ad',           'label_col': 'cell_type',                'snrna_seq': False},
]

TOP_K_GENES        = 200
MAX_CELLS_PER_TYPE = 300   # default if a cfg doesn't specify
MIN_CELLS_PER_TYPE = 100
TEST_FRAC          = 0.15
VAL_FRAC           = 0.10
BATCH_SIZE         = 16
N_EPOCHS           = 10
LEARNING_RATE      = 1e-4
WEIGHT_DECAY       = 1e-2
WARMUP_STEPS       = 200
MAX_SEQ_LEN        = 512
MAX_WEIGHT         = 20.0   # v8: raised from 10 to let inverse-frequency boost reach minority classes
SEED               = 42

# Targeted post-clamp weight boosts for lineage-critical minority classes (v8)
TARGETED_WEIGHT_BOOSTS = {
    'CD4-positive, alpha-beta T cell':       1.5,
    'regulatory T cell':                     1.5,
    'classical monocyte':                    1.5,
    'non-classical monocyte':                1.5,
    'monocyte':                              1.5,
}

# Inference-time hyperparameters (set TEMPERATURE after fitting; thresholds for roll-up)
TAU_LEAF   = 0.50   # if max leaf prob ≥ this, output the leaf
TAU_PARENT = 0.70   # else, output deepest ancestor with prob ≥ this

os.makedirs(OUT_DIR, exist_ok=True)
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'  Device : {device}  |  Output dir : {OUT_DIR}')
print(f'  ORGAN_CONFIGS: {len(ORGAN_CONFIGS)} training organs')
print(f'  LAB_CONFIGS  : {len(LAB_CONFIGS)} zero-shot datasets')
print(f'  PER_LEAF_MAX_BY_ORGAN: {sum(len(v) for v in PER_LEAF_MAX_BY_ORGAN.values())} subtype caps')
print(f'  MAX_WEIGHT={MAX_WEIGHT}  |  targeted boosts={len(TARGETED_WEIGHT_BOOSTS)}')
print('[OK] Config ready')


## 3. Data Loading & Preprocessing

In [ ]:
def is_valid(value):
    if pd.isna(value): return False
    return str(value).strip().lower() not in ('', 'nan', 'none', 'unknown', 'na', 'n/a')

def get_gene_symbols(adata):
    if 'feature_name' in adata.var.columns:
        return np.array(adata.var['feature_name'].astype(str))
    return np.array(adata.var_names.astype(str))

# ── Gene filter: remove snRNA-seq / housekeeping artifacts ───────────────────
EXCLUDE_PREFIXES = ('MT-', 'MTRNR', 'RPS', 'RPL', 'LINC', 'MIR',
                    'SNORD', 'SNORA', 'SNHG', 'ENSG', 'AC0', 'AC1',
                    'AL0', 'AL1', 'AP0', 'LOC')
EXCLUDE_EXACT    = {'MALAT1', 'NEAT1', 'XIST', 'TSIX', 'KCNQ1OT1',
                    'FTX', 'JPX', 'NORAD', 'HOTAIR', 'SOX2-OT',
                    'TUG1', 'GAS5', 'HOTAIRM1', 'PVT1'}
EXCLUDE_SUFFIXES = ('-AS1', '-AS2', '-AS3', '-AS4', '-AS5',
                    '-IT1', '-IT2', '-OT', '-OT1', '-OT2', '-OT3')

def is_keep_gene(g):
    g = str(g)
    if g in EXCLUDE_EXACT: return False
    if g.startswith(EXCLUDE_PREFIXES): return False
    if g.endswith(EXCLUDE_SUFFIXES): return False
    return True

def keep_gene_mask(gene_symbols):
    return np.array([is_keep_gene(g) for g in gene_symbols], dtype=bool)

def get_gene_lengths(adata, gene_symbols, default=2000.0):
    """Per-gene length from feature_length column (default fallback if missing)."""
    n = len(gene_symbols)
    if 'feature_length' in adata.var.columns:
        lengths = pd.to_numeric(adata.var['feature_length'], errors='coerce').values.astype(float)
        return np.where((lengths > 0) & np.isfinite(lengths), lengths, default)
    return np.full(n, default, dtype=float)


def load_organ(cfg, min_cells=MIN_CELLS_PER_TYPE, max_cells=None):
    if max_cells is None:
        max_cells = cfg.get('max_cells', MAX_CELLS_PER_TYPE)
    name, label_col = cfg['name'], cfg['label_col']
    t0 = time.time()
    adata = sc.read_h5ad(cfg['path'], backed='r')
    gene_symbols = get_gene_symbols(adata)
    valid_mask = adata.obs[label_col].apply(is_valid)
    obs        = adata.obs[valid_mask].copy()
    valid_idx  = np.where(valid_mask.values)[0]
    labels  = obs[label_col].astype(str).values
    # ── v8: per-leaf cap dict (e.g. lung fibroblast subtypes) overrides organ cap ─
    per_leaf_cap = PER_LEAF_MAX_BY_ORGAN.get(name, {})
    sampled = []
    capped_subtypes = []
    for lbl in np.unique(labels):
        pos = np.where(labels == lbl)[0]
        lbl_cap = per_leaf_cap.get(lbl, max_cells)
        if len(pos) > lbl_cap:
            pos = np.random.choice(pos, lbl_cap, replace=False)
            if lbl in per_leaf_cap:
                capped_subtypes.append((lbl, lbl_cap))
        sampled.extend(pos.tolist())
    if capped_subtypes:
        for lbl, cap in capped_subtypes:
            print(f'  [{name}] subtype-cap "{lbl}" → {cap}')
    sampled   = np.sort(np.array(sampled, dtype=np.int64))
    obs       = obs.iloc[sampled].copy()
    valid_idx = valid_idx[sampled]
    counts  = obs[label_col].value_counts()
    keep    = counts[counts >= min_cells].index
    dropped = counts[counts < min_cells]
    if len(dropped):
        print(f'  [{name}] Dropping {len(dropped)} type(s) < {min_cells} cells')
    mask      = obs[label_col].isin(keep)
    obs       = obs[mask].copy()
    valid_idx = valid_idx[mask.values]
    print(f'  [{name}] {len(obs):,} cells | {obs[label_col].nunique()} types | max_cells={max_cells} | snrna_seq={cfg.get("snrna_seq", False)} | {time.time()-t0:.1f}s')
    return adata, obs, valid_idx, gene_symbols


def cell_to_text_backed(h5_path, row_indices, gene_symbols, top_k=200, desc='cells',
                        gene_lengths=None, length_normalize=False):
    gene_keep = keep_gene_mask(gene_symbols)
    if length_normalize and gene_lengths is None:
        raise ValueError('length_normalize=True requires gene_lengths')
    texts = []
    with h5py.File(h5_path, 'r') as f:
        X = f['X']
        indptr = X['indptr'][:]
        indices_ds, data_ds = X['indices'], X['data']
        for row_idx in tqdm(row_indices, desc=f'  {desc}', leave=False):
            start, end = int(indptr[row_idx]), int(indptr[row_idx + 1])
            if start == end: texts.append(''); continue
            vals = data_ds[start:end]
            cols = indices_ds[start:end]
            keep = gene_keep[cols]; vals = vals[keep]; cols = cols[keep]
            if len(vals) == 0: texts.append(''); continue
            score = vals / gene_lengths[cols] if length_normalize else vals
            if len(score) <= top_k:
                order = np.argsort(score)[::-1]
            else:
                order = np.argpartition(score, -top_k)[-top_k:]
                order = order[np.argsort(score[order])[::-1]]
            texts.append(' '.join(str(gene_symbols[cols[j]]) for j in order if vals[j] > 0))
    return texts


def cell_to_text_dense(X_row, gene_symbols, top_k=200, gene_keep=None,
                       gene_lengths=None, length_normalize=False):
    if gene_keep is None:
        gene_keep = keep_gene_mask(gene_symbols)
    row = X_row.toarray().flatten() if sp.issparse(X_row) else np.array(X_row).flatten()
    nz  = np.where(row > 0)[0]
    if len(nz) == 0: return ''
    nz = nz[gene_keep[nz]]
    if len(nz) == 0: return ''
    vals  = row[nz]
    score = (vals / gene_lengths[nz]) if (length_normalize and gene_lengths is not None) else vals
    if len(nz) > top_k:
        idx = np.argpartition(score, -top_k)[-top_k:]
        nz  = nz[idx[np.argsort(score[idx])[::-1]]]
    else:
        nz = nz[np.argsort(score)[::-1]]
    return ' '.join(gene_symbols[i] for i in nz)


def cell_to_text_robust(h5_path, row_indices, gene_symbols, top_k=200, desc='cells',
                        gene_lengths=None, length_normalize=False):
    try:
        return cell_to_text_backed(h5_path, row_indices, gene_symbols, top_k, desc,
                                    gene_lengths=gene_lengths, length_normalize=length_normalize)
    except (KeyError, AttributeError, TypeError, OSError):
        print(f'  [{desc}] CSR streaming failed — loading X into memory')
        adata_tmp = sc.read_h5ad(h5_path)
        X = adata_tmp.X
        gene_keep = keep_gene_mask(gene_symbols)
        texts = [cell_to_text_dense(X[i], gene_symbols, top_k, gene_keep=gene_keep,
                                     gene_lengths=gene_lengths, length_normalize=length_normalize)
                 for i in tqdm(row_indices, desc=f'  {desc}', leave=False)]
        del adata_tmp
        return texts

print('[OK] Helpers ready (with gene filter + length norm + per-organ + per-leaf caps)')

In [ ]:
print('Loading all organs ...')
loaded_organs = {}
for cfg in ORGAN_CONFIGS:
    adata, obs, valid_idx, gene_syms = load_organ(cfg)
    loaded_organs[cfg['name']] = {'cfg': cfg, 'adata': adata, 'obs': obs,
                                   'valid_idx': valid_idx, 'gene_symbols': gene_syms}

# ── Label normalization: disease filter (lymphoid/myeloid removed in v7) ─────
for organ_name in ['liver', 'lymph_node', 'bone_marrow']:
    d = loaded_organs[organ_name]
    obs = d['obs']
    if 'disease' in obs.columns:
        non_normal = (obs['disease'] != 'normal').sum()
        if non_normal:
            mask = obs['disease'] == 'normal'
            d['obs'] = obs[mask].copy()
            d['valid_idx'] = d['valid_idx'][mask.values]
            print(f'  [{organ_name}] Removed {non_normal} non-normal cells')

# Fix lung pericyte naming
loaded_organs['lung']['obs']['ann_finest_level'] = (
    loaded_organs['lung']['obs']['ann_finest_level'].replace('Pericytes', 'pericyte')
)

print('[OK] All organs loaded and normalized')

In [ ]:
LABEL_SYNONYM_MAP = {
    'B cells': 'B cell', 'NK cells': 'natural killer cell',
    'Alveolar macrophages': 'alveolar macrophage', 'Mast cells': 'mast cell',
    'Plasma cells': 'plasma cell', 'Classical monocytes': 'classical monocyte',
    'Non-classical monocytes': 'non-classical monocyte',
    'Plasmacytoid DCs': 'plasmacytoid dendritic cell',
    'Smooth muscle': 'smooth muscle cell',
    'CD4 T cells': 'CD4-positive, alpha-beta T cell',
    'CD4-positive helper T cell': 'CD4-positive, alpha-beta T cell',
    'CD8 T cells': 'CD8-positive, alpha-beta T cell',
    'CD4-positive, CD25-positive, alpha-beta regulatory T cell': 'regulatory T cell',
    'CD8-positive, alpha-beta memory T cell, CD45RO-positive': 'CD8-positive, alpha-beta memory T cell',
    'mature alpha-beta T cell': 'alpha-beta T cell',
    'mature NK T cell': 'natural killer T cell',
    'mature B cell': 'B cell',
    'effector memory CD4-positive, alpha-beta T cell, terminally differentiated': 'effector memory CD4-positive, alpha-beta T cell',
    'dendritic cell, human': 'dendritic cell',
    'group 3 innate lymphoid cell, human': 'group 3 innate lymphoid cell',
    'plasmacytoid dendritic cell, human': 'plasmacytoid dendritic cell',
    'CD14-positive monocyte': 'classical monocyte',
    'CD14-positive, CD16-positive monocyte': 'intermediate monocyte',
    'myeloid dendritic cell': 'conventional dendritic cell',
    'liver dendritic cell': 'conventional dendritic cell',
    'vein endothelial cell': 'endothelial cell of vein',
    'endothelial cell of pericentral hepatic sinusoid': 'endothelial cell of hepatic sinusoid',
    'endothelial cell of periportal hepatic sinusoid': 'endothelial cell of hepatic sinusoid',
    'intrahepatic cholangiocyte': 'cholangiocyte',
    'granulocyte monocyte progenitor cell': 'granulocyte monocyte progenitor',
    'cycling plasma cell': 'plasma cell',
    'inflammatory macrophage': 'macrophage',
    'B_cell': 'B cell', 'B_cell_naive': 'naive B cell',
    'NK_cell': 'natural killer cell', 'plasma_cell': 'plasma cell',
    'plasma_cell_proliferating': 'plasma cell', 'Treg_cell': 'regulatory T cell',
    'alveolar_macrophage': 'alveolar macrophage', 'mast_cell': 'mast cell',
    'monocyte_CSF3R': 'classical monocyte', 'monocyte_SOCS3': 'classical monocyte',
    'monocyte_ITGAL': 'non-classical monocyte', 'monocyte_AREG_EREG': 'monocyte',
    'cDC1': 'conventional dendritic cell',
}

# Bad label filters
BAD_LABEL_FILTERS = [
    ('liver', 'cell_type', {'malignant cell'}),
    ('lymph_node', 'cell_type', {'stromal cell of pancreas', 'alveolar macrophage'}),
]
for organ_name, col, bad_labels in BAD_LABEL_FILTERS:
    d, obs = loaded_organs[organ_name], loaded_organs[organ_name]['obs']
    for lbl in bad_labels:
        n = (obs[col] == lbl).sum()
        if n:
            mask = obs[col] != lbl
            d['obs'] = obs[mask].copy()
            d['valid_idx'] = d['valid_idx'][mask.values]
            obs = d['obs']
            print(f'  [{organ_name}] Dropped "{lbl}" ({n} cells)')

# Apply synonyms
total_remapped = 0
for cfg in ORGAN_CONFIGS:
    name, col = cfg['name'], cfg['label_col']
    obs = loaded_organs[name]['obs']
    for src, tgt in LABEL_SYNONYM_MAP.items():
        n = (obs[col] == src).sum()
        if n:
            loaded_organs[name]['obs'][col] = obs[col].replace(src, tgt)
            obs = loaded_organs[name]['obs']
            total_remapped += n
print(f'[OK] Synonyms applied — {total_remapped:,} cells remapped')

In [ ]:
# ── Lung hierarchy ────────────────────────────────────────────────────────────
lung_cfg  = loaded_organs['lung']['cfg']
lung_obs2 = loaded_organs['lung']['obs']
lung_ontology = {}
level_cols = [c for c in lung_cfg['hierarchy_cols'] if c in lung_obs2.columns]
cols_ordered = level_cols + [lung_cfg['label_col']]
for k in range(1, len(cols_ordered)):
    parent_col, child_col = cols_ordered[k-1], cols_ordered[k]
    for _, row in lung_obs2[[parent_col, child_col]].dropna().drop_duplicates().iterrows():
        p, ch = str(row[parent_col]), str(row[child_col])
        if is_valid(p) and is_valid(ch) and ch not in lung_ontology:
            lung_ontology[ch] = p
for val in lung_obs2[cols_ordered[0]].dropna().unique():
    if is_valid(val) and str(val) not in lung_ontology:
        lung_ontology[str(val)] = None

# ── Brain glia hierarchy ──────────────────────────────────────────────────────
brain_glia_cfg = loaded_organs['brain_glia']['cfg']
brain_glia_obs = loaded_organs['brain_glia']['obs']
brain_glia_ontology = {}
coarse_col = brain_glia_cfg['coarse_col']
for _, row in brain_glia_obs[[coarse_col, brain_glia_cfg['label_col']]].dropna().drop_duplicates().iterrows():
    brain_glia_ontology[str(row[brain_glia_cfg['label_col']])] = str(row[coarse_col])
for val in brain_glia_obs[coarse_col].dropna().unique():
    if str(val) not in brain_glia_ontology:
        brain_glia_ontology[str(val)] = None

# ── Cross-organ hierarchy (v6) ────────────────────────────────────────────────
CROSS_ORGAN_HIERARCHY = {
    'Upper-layer intratelencephalic': 'excitatory neuron', 'Deep-layer intratelencephalic': 'excitatory neuron',
    'Deep-layer corticothalamic and 6b': 'excitatory neuron', 'Deep-layer near-projecting': 'excitatory neuron',
    'Hippocampal CA1-3': 'excitatory neuron', 'Hippocampal CA4': 'excitatory neuron',
    'Hippocampal dentate gyrus': 'excitatory neuron', 'Thalamic excitatory': 'excitatory neuron',
    'Amygdala excitatory': 'excitatory neuron', 'Upper rhombic lip': 'excitatory neuron',
    'Lower rhombic lip': 'excitatory neuron', 'Mammillary body': 'excitatory neuron',
    'CGE interneuron': 'inhibitory neuron', 'MGE interneuron': 'inhibitory neuron',
    'Cerebellar inhibitory': 'inhibitory neuron', 'LAMP5-LHX6 and Chandelier': 'inhibitory neuron',
    'Medium spiny neuron': 'inhibitory neuron', 'Eccentric medium spiny neuron': 'inhibitory neuron',
    'Midbrain-derived inhibitory': 'inhibitory neuron', 'Miscellaneous': 'neuron',
    'excitatory neuron': 'neuron', 'inhibitory neuron': 'neuron',
    'CD4-positive, alpha-beta T cell': 'alpha-beta T cell', 'CD8-positive, alpha-beta T cell': 'alpha-beta T cell',
    'regulatory T cell': 'CD4-positive, alpha-beta T cell', 'T follicular helper cell': 'CD4-positive, alpha-beta T cell',
    'naive thymus-derived CD4-positive, alpha-beta T cell': 'CD4-positive, alpha-beta T cell',
    'central memory CD4-positive, alpha-beta T cell': 'CD4-positive, alpha-beta T cell',
    'effector memory CD4-positive, alpha-beta T cell': 'CD4-positive, alpha-beta T cell',
    'naive thymus-derived CD8-positive, alpha-beta T cell': 'CD8-positive, alpha-beta T cell',
    'central memory CD8-positive, alpha-beta T cell': 'CD8-positive, alpha-beta T cell',
    'effector memory CD8-positive, alpha-beta T cell': 'CD8-positive, alpha-beta T cell',
    'effector CD8-positive, alpha-beta T cell': 'CD8-positive, alpha-beta T cell',
    'gamma-delta T cell': 'T cell', 'mucosal invariant T cell': 'T cell',
    'natural killer T cell': 'T cell', 'alpha-beta T cell': 'T cell', 'T cell': 'lymphocyte',
    'naive B cell': 'B cell', 'memory B cell': 'B cell', 'germinal center B cell': 'B cell',
    'plasmablast': 'B cell', 'plasma cell': 'B cell', 'transitional stage B cell': 'B cell',
    'B cell': 'lymphocyte', 'natural killer cell': 'lymphocyte', 'innate lymphoid cell': 'lymphocyte',
    'group 1 innate lymphoid cell': 'innate lymphoid cell', 'group 2 innate lymphoid cell': 'innate lymphoid cell',
    'group 3 innate lymphoid cell': 'innate lymphoid cell', 'lymphocyte': 'leukocyte',
    'classical monocyte': 'monocyte', 'non-classical monocyte': 'monocyte',
    'intermediate monocyte': 'monocyte', 'monocyte': 'myeloid leukocyte',
    'Kupffer cell': 'macrophage', 'alveolar macrophage': 'macrophage', 'macrophage': 'myeloid leukocyte',
    'conventional dendritic cell': 'dendritic cell', 'plasmacytoid dendritic cell': 'dendritic cell',
    'dendritic cell': 'myeloid leukocyte', 'mast cell': 'myeloid leukocyte',
    'neutrophil': 'myeloid leukocyte', 'basophil': 'myeloid leukocyte',
    'eosinophil': 'myeloid leukocyte', 'myeloid leukocyte': 'leukocyte', 'leukocyte': 'Immune',
    'hematopoietic stem cell': 'hematopoietic precursor cell',
    'common myeloid progenitor': 'hematopoietic precursor cell',
    'granulocyte monocyte progenitor': 'hematopoietic precursor cell',
    'common lymphoid progenitor': 'hematopoietic precursor cell',
    'hematopoietic precursor cell': 'Immune',
    'proerythroblast': 'erythroid lineage cell', 'erythroblast': 'erythroid lineage cell',
    'reticulocyte': 'erythroid lineage cell', 'erythrocyte': 'erythroid lineage cell',
    'erythroid lineage cell': 'hematopoietic precursor cell',
    'megakaryocyte-erythroid progenitor cell': 'hematopoietic precursor cell',
    'megakaryocyte': 'hematopoietic precursor cell', 'platelet': 'megakaryocyte',
    'endothelial cell of artery': 'endothelial cell', 'endothelial cell of vein': 'endothelial cell',
    'endothelial cell of hepatic sinusoid': 'endothelial cell',
    'blood vessel endothelial cell': 'endothelial cell', 'lymphatic endothelial cell': 'endothelial cell',
    'high endothelial venule cell': 'endothelial cell', 'capillary endothelial cell': 'endothelial cell',
    'hepatic stellate cell': 'fibroblast', 'portal fibroblast': 'fibroblast',
    'fibroblastic reticular cell': 'fibroblast',
    'hepatocyte': 'hepatic cell', 'cholangiocyte': 'hepatic cell', 'hepatic cell': 'Epithelial',
    'Oligodendrocyte': 'glial cell', 'Astrocyte': 'glial cell', 'Microglia': 'glial cell',
    'Committed oligodendrocyte precursor': 'glial cell', 'Oligodendrocyte precursor': 'glial cell',
    'Bergmann glia': 'glial cell', 'Choroid plexus': 'glial cell', 'Ependymal': 'glial cell',
    'Fibroblast': 'Fibroblast lineage',
    # Step 1: wired atlas labels
    'Interstitial Mph perivascular': 'macrophage', 'Peribronchial fibroblasts': 'fibroblast',
    'EC general capillary': 'capillary endothelial cell', 'EC venous pulmonary': 'endothelial cell of vein',
    'Lymphatic EC mature': 'lymphatic endothelial cell',
    'central nervous system macrophage': 'macrophage', 'microglial cell': 'macrophage',
    # Step 2a: lymphoid activation state leaves
    'CD4_T_cell_activated': 'CD4-positive, alpha-beta T cell',
    'CD4_T_cell_naive_or_memory': 'CD4-positive, alpha-beta T cell',
    'CD8_T_cell_early_activated': 'CD8-positive, alpha-beta T cell',
    'CD8_T_cell_late_exhausted': 'CD8-positive, alpha-beta T cell',
    'CD8_T_cell_proliferating': 'CD8-positive, alpha-beta T cell',
    # Step 2b: myeloid macrophage functional state leaves
    'macrophage_APOE_CHIT': 'macrophage', 'macrophage_C3': 'macrophage',
    'macrophage_F13A1': 'macrophage', 'macrophage_FOLR2': 'macrophage',
    'macrophage_ISG_expressing': 'macrophage', 'macrophage_VEGFA': 'macrophage',
    'macrophage_glycolytic': 'macrophage',
    # Step 3: neuron fallback
    'neuron (unspecified)': 'neuron', 'GABAergic_interneuron': 'inhibitory neuron',
    'GABAergic_interneuron_SST': 'MGE interneuron', 'astrocyte_fibrous_like': 'Astrocyte',
    'astrocyte_protoplasmic': 'Astrocyte', 'arterial_endothelial_cell': 'endothelial cell of artery',
    'capillary_endothelial_cell': 'capillary endothelial cell',
    'venous_endothelial_cell': 'endothelial cell of vein',
    'meningeal_fibroblast': 'fibroblast', 'perivascular_fibroblast': 'fibroblast',
    'smooth_muscle_cell': 'smooth muscle cell', 'oligodendrocyte_ISG_expressing': 'Oligodendrocyte',
    'oligodendrocyte_precursor_cell': 'Oligodendrocyte precursor',
}

# Merge ontologies
combined_ontology = {}
combined_ontology.update(lung_ontology)
combined_ontology.update(brain_glia_ontology)
combined_ontology.update(CROSS_ORGAN_HIERARCHY)
for organ_name in ['liver', 'lymph_node', 'bone_marrow']:
    d = loaded_organs[organ_name]
    for ct in d['obs'][d['cfg']['label_col']].unique():
        ct = str(ct)
        if ct not in combined_ontology:
            combined_ontology[ct] = None
combined_ontology['pericyte'] = 'Vascular'

# Deduplicate collisions
for cfg in ORGAN_CONFIGS:
    obs, combined_ontology, report = deduplicate_hierarchy(
        loaded_organs[cfg['name']]['obs'], combined_ontology, cfg['label_col']
    )
    loaded_organs[cfg['name']]['obs'] = obs
    if not report.empty:
        for _, row in report.iterrows():
            print(f'  [{cfg["name"]}] "{row["original_label"]}" -> "{row["new_label"]}" ({row["n_cells_renamed"]})')

# Ensure synonym targets are in ontology
for tgt in set(LABEL_SYNONYM_MAP.values()):
    if tgt not in combined_ontology:
        combined_ontology[tgt] = None

print(f'[OK] Combined ontology: {len(combined_ontology)} entries')

In [ ]:
print('Cell-to-text conversion ...')
CSR_ORGANS = {'lung', 'brain_glia', 'brain_neurons', 'liver', 'lymph_node', 'bone_marrow'}
all_texts, all_labels_str, all_organ_ids = [], [], []
organ_texts, organ_labels = {}, {}

for cfg in ORGAN_CONFIGS:
    name = cfg['name']
    d    = loaded_organs[name]
    gene_lengths = get_gene_lengths(d['adata'], d['gene_symbols'])
    length_norm  = cfg.get('snrna_seq', False)
    print(f'  [{name}] {len(d["valid_idx"]):,} cells | length_normalize={length_norm} ...')
    fn = cell_to_text_backed if name in CSR_ORGANS else cell_to_text_robust
    texts = fn(cfg['path'], d['valid_idx'], d['gene_symbols'], TOP_K_GENES,
               desc=name, gene_lengths=gene_lengths, length_normalize=length_norm)
    keep_mask = [bool(t.strip()) for t in texts]
    texts     = [t for t, k in zip(texts, keep_mask) if k]
    labels    = d['obs'][cfg['label_col']].astype(str).values[keep_mask]
    organ_texts[name], organ_labels[name] = texts, labels
    all_texts.extend(texts)
    all_labels_str.extend(labels)
    all_organ_ids.extend([cfg['id']] * len(texts))

all_labels_str = np.array(all_labels_str)
all_organ_ids  = np.array(all_organ_ids, dtype=int)
print(f'[OK] {len(all_texts):,} cells | {len(np.unique(all_labels_str))} unique labels')

In [ ]:
print('Building vocabulary and splits ...')
tokenizer = AutoTokenizer.from_pretrained(C2S_MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

leaf_classes_set = set(all_labels_str)
all_nodes = set()
for child, parent in combined_ontology.items():
    all_nodes.add(child)
    if parent: all_nodes.add(parent)
all_nodes |= leaf_classes_set

class_names   = sorted(all_nodes)
class_to_idx  = {name: idx for idx, name in enumerate(class_names)}
n_classes     = len(class_names)
leaf_classes  = sorted(leaf_classes_set)
leaf_indices  = [class_to_idx[c] for c in leaf_classes]
leaf_index_set = set(leaf_indices)

# Leaf-local index mapping (for CE baseline)
leaf_to_local   = {leaf_idx: i for i, leaf_idx in enumerate(leaf_indices)}
local_to_leaf   = {i: leaf_idx for i, leaf_idx in enumerate(leaf_indices)}
n_leaf_classes  = len(leaf_classes)

labels_encoded = np.array([class_to_idx[s] for s in all_labels_str], dtype=int)

class CellTextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.texts, self.labels = texts, labels
        self.tokenizer, self.max_length = tokenizer, max_length
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx], truncation=True, max_length=self.max_length,
            padding='max_length', return_tensors='pt')
        return {'input_ids': enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0),
                'label': torch.tensor(self.labels[idx], dtype=torch.long)}

strat_key = labels_encoded * 10 + all_organ_ids
idx_all   = np.arange(len(all_texts))
idx_tv, idx_test   = train_test_split(idx_all, test_size=TEST_FRAC, stratify=strat_key, random_state=SEED)
idx_train, idx_val = train_test_split(idx_tv,  test_size=VAL_FRAC/(1-TEST_FRAC), stratify=strat_key[idx_tv], random_state=SEED)

train_labels = labels_encoded[idx_train]
val_labels   = labels_encoded[idx_val]
test_labels  = labels_encoded[idx_test]
test_organ   = all_organ_ids[idx_test]

train_ds = CellTextDataset([all_texts[i] for i in idx_train], train_labels, tokenizer, MAX_SEQ_LEN)
val_ds   = CellTextDataset([all_texts[i] for i in idx_val],   val_labels,   tokenizer, MAX_SEQ_LEN)
test_ds  = CellTextDataset([all_texts[i] for i in idx_test],  test_labels,  tokenizer, MAX_SEQ_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'  Total vocab : {n_classes}  |  Leaf classes : {n_leaf_classes}')
print(f'  Train : {len(idx_train):,} | Val : {len(idx_val):,} | Test : {len(idx_test):,}')
print('[OK] Vocabulary and splits ready')

## 3.5 Diagnostic: training-set class counts for lineage-critical labels


In [ ]:
# v8 diagnostic: surfaces actual CD4/CD8/monocyte/macrophage counts AFTER all
# label normalization + caps + min-cells filter. Use this to decide if the
# targeted weight boosts in Cell 4 are sufficient (or if data needs rebalancing).
from collections import Counter
counts = Counter(class_names[i] for i in train_labels)

KEYS_OF_INTEREST = [
    'CD4-positive, alpha-beta T cell',
    'CD8-positive, alpha-beta T cell',
    'regulatory T cell',
    'classical monocyte',
    'non-classical monocyte',
    'monocyte',
    'macrophage',
    'macrophage (unspecified)',
    'alveolar macrophage',
    'plasma cell',
    'B cell',
    'natural killer cell',
    'neuron (unspecified)',
    'oligodendrocyte',
    'astrocyte',
]
print('  v8 training counts for lineage-critical classes:')
print(f'  {"Class":<48} {"count":>8}')
print('  ' + '-' * 58)
for k in KEYS_OF_INTEREST:
    print(f'  {k:<48} {counts.get(k, 0):>8}')

cd4 = counts.get('CD4-positive, alpha-beta T cell', 0)
cd8 = counts.get('CD8-positive, alpha-beta T cell', 0)
if cd4 and cd8:
    ratio = cd8 / max(cd4, 1)
    if ratio >= 2.0:
        print(f'\n  ⚠ CD8/CD4 ratio = {ratio:.2f}× — TARGETED_WEIGHT_BOOSTS in Cell 4 should help.')
        print(f'    If still biased after retrain, downsample CD8 in load step or boost CD4 further.')
    else:
        print(f'\n  CD8/CD4 ratio = {ratio:.2f}× — balanced.')

mono = counts.get('classical monocyte', 0) + counts.get('non-classical monocyte', 0) + counts.get('monocyte', 0)
mac  = counts.get('macrophage', 0) + counts.get('macrophage (unspecified)', 0) + counts.get('alveolar macrophage', 0)
if mono and mac:
    print(f'  monocyte total ≈ {mono}   macrophage total ≈ {mac}   ratio mac/mono = {mac/max(mono,1):.2f}')
    if mono < 200:
        print(f'  ⚠ Monocyte training cells < 200 — predictions may default to macrophage.')


## 4. Loss Function & Class Weights *(HCE only)*

In [ ]:
print('Building reachability matrix and class weights ...')

# ── Reachability matrix ──────────────────────────────────────────────────────
R_np = build_reachability_matrix_from_ontology(combined_ontology, class_names)
reachability_matrix = torch.tensor(R_np, dtype=torch.float32).to(device)
print(f'  R shape : {n_classes}x{n_classes} | nnz={int(reachability_matrix.sum())}')

# ── Class weights: w_i = N / (C * n_i), ancestors get effective-count weight ─
train_counts   = Counter(train_labels.tolist())
N_train, C_obs = len(train_labels), len(train_counts)

class_weights_full = torch.zeros(n_classes, dtype=torch.float32, device=device)
for idx_ct, count in train_counts.items():
    class_weights_full[idx_ct] = N_train / (C_obs * count)
ancestor_indices = [i for i in range(n_classes) if i not in leaf_index_set]
for anc_idx in ancestor_indices:
    eff = sum(train_counts.get(j, 0) for j in leaf_indices if R_np[anc_idx, j] > 0)
    if eff > 0:
        class_weights_full[anc_idx] = N_train / (C_obs * eff)
class_weights_full = torch.clamp(class_weights_full, max=MAX_WEIGHT)

# ── v8: targeted post-clamp boosts for lineage-critical minority classes ─────
boosted = []
for cls_name, factor in TARGETED_WEIGHT_BOOSTS.items():
    idx_cls = class_to_idx.get(cls_name, -1)
    if idx_cls >= 0 and class_weights_full[idx_cls] > 0:
        class_weights_full[idx_cls] = class_weights_full[idx_cls] * factor
        boosted.append((cls_name, factor, float(class_weights_full[idx_cls])))
for nm, f, w in boosted:
    print(f'  boost {f}× → {nm:<45} weight={w:.3f}')

# ── HCE loss ─────────────────────────────────────────────────────────────────
class HCELoss(nn.Module):
    """Hierarchical cross-entropy: s = softmax(logits) @ R^T, loss = -w_t * log(s_t)."""
    def __init__(self, R, class_weights=None, eps=1e-8):
        super().__init__()
        self.register_buffer('R', R)
        self.eps = eps
        if class_weights is not None:
            self.register_buffer('class_weights', class_weights)
        else:
            self.class_weights = None
    def forward(self, logits, targets):
        probs  = torch.softmax(logits, dim=1)
        s      = torch.clamp(probs @ self.R.T, min=self.eps)
        log_st = torch.log(s)[torch.arange(len(targets), device=targets.device), targets]
        if self.class_weights is not None:
            return -(self.class_weights[targets] * log_st).mean()
        return -log_st.mean()

hce_criterion = HCELoss(reachability_matrix, class_weights_full).to(device)
print('[OK] HCE loss ready')
print(f'  HCE class weights : {(class_weights_full > 0).sum().item()}/{n_classes} non-zero')

## 4.5 S1-D Diagnostic — which `TARGETED_WEIGHT_BOOSTS` actually applied (post-dedup)


In [ ]:
# v9 Stage 1, S1-D: confirm whether v8 boosts hit their intended classes,
# or were silently dropped because deduplicate_hierarchy() renamed those classes
# to "X (unspecified)". Diagnostic-only — does not alter weights.
print('  S1-D: boost-target resolution check')
print(f'  {"base name":<46}  {"as-is in class_to_idx":<24}  {"unspec version exists":<22}  {"verdict"}')
print('  ' + '-' * 110)
silent_no_ops = []
applied = []
for base, factor in TARGETED_WEIGHT_BOOSTS.items():
    as_is_present  = base in class_to_idx
    unspec_name    = f'{base} (unspecified)'
    unspec_present = unspec_name in class_to_idx
    if as_is_present:
        verdict = 'BOOST APPLIED'
        applied.append(base)
    elif unspec_present:
        verdict = 'SILENT NO-OP (renamed)'
        silent_no_ops.append((base, unspec_name))
    else:
        verdict = 'NOT FOUND'
        silent_no_ops.append((base, '<missing>'))
    print(f'  {base:<46}  {str(as_is_present):<24}  {str(unspec_present):<22}  {verdict}')

print(f'\n  Applied boosts ({len(applied)}): {applied}')
if silent_no_ops:
    print(f'  Silent no-ops ({len(silent_no_ops)}):')
    for base, actual in silent_no_ops:
        print(f'    "{base}"  →  actually in vocab as  "{actual}"')
    print('  → In Stage 2 retrain, update TARGETED_WEIGHT_BOOSTS keys OR resolve at runtime.')
else:
    print('  All boosts hit their intended targets — no Stage 2 boost-key fix needed.')


## 5. Model Architecture

In [ ]:
class C2SClassifier(nn.Module):
    """C2S-Pythia-410m encoder + dropout + linear head over n_classes."""
    def __init__(self, encoder, hidden_size, num_classes):
        super().__init__()
        self.encoder = encoder
        self.dropout = nn.Dropout(0.1)
        self.head    = nn.Linear(hidden_size, num_classes)
    def forward(self, input_ids, attention_mask):
        out         = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden = out.last_hidden_state
        seq_len     = attention_mask.sum(dim=1) - 1
        last_token  = last_hidden[torch.arange(last_hidden.size(0), device=last_hidden.device), seq_len]
        return self.head(self.dropout(last_token))

def build_fresh_model():
    """Load a fresh C2S encoder and wrap in C2SClassifier."""
    enc = AutoModel.from_pretrained(C2S_MODEL_NAME)
    enc.gradient_checkpointing_enable()
    return C2SClassifier(enc, enc.config.hidden_size, n_classes).to(device)

def build_optimizer_and_schedulers(model, n_steps):
    opt = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    warmup = optim.lr_scheduler.LinearLR(opt, start_factor=0.1, total_iters=WARMUP_STEPS)
    cosine = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, n_steps - WARMUP_STEPS))
    return opt, warmup, cosine

total_params = sum(p.numel() for p in build_fresh_model().parameters()) / 1e6
print(f'[OK] Architecture: C2S-Pythia-410m + head  |  {total_params:.1f}M params  |  {n_classes} output classes')

## 6. Training Loop

In [ ]:
leaf_indices_t = torch.tensor(leaf_indices, device=device)

def train_model(model, criterion, save_path, label):
    """
    Train model for N_EPOCHS, save best checkpoint by val accuracy.
    Returns history dict and best_val_acc.
    """
    total_steps = len(train_loader) * N_EPOCHS
    optimizer, warmup_sched, cosine_sched = build_optimizer_and_schedulers(model, total_steps)
    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    best_val_acc, global_step = 0.0, 0

    print(f'\n{"="*60}')
    print(f'  Training [{label}]')
    print(f'  Steps/epoch: {len(train_loader)}  |  Total: {total_steps}')
    print('=' * 60)
    t_start = time.time()

    for epoch in range(1, N_EPOCHS + 1):
        t_epoch = time.time()
        model.train()
        running_loss, n_batches = 0.0, 0
        pbar = tqdm(train_loader, desc=f'  [{label}] Epoch {epoch}/{N_EPOCHS} [train]', leave=True)
        for batch in pbar:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels_b       = batch['label'].to(device)
            optimizer.zero_grad()
            logits = model(input_ids, attention_mask)
            loss   = criterion(logits, labels_b)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            if global_step < WARMUP_STEPS: warmup_sched.step()
            else: cosine_sched.step()
            global_step += 1
            running_loss += loss.item()
            n_batches    += 1
            pbar.set_postfix(loss=f'{loss.item():.4f}', lr=f'{optimizer.param_groups[0]["lr"]:.2e}')
        train_loss = running_loss / n_batches

        model.eval()
        val_loss_sum, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f'  [{label}] Epoch {epoch}/{N_EPOCHS} [val]  ', leave=False):
                input_ids      = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels_b       = batch['label'].to(device)
                logits         = model(input_ids, attention_mask)
                val_loss_sum  += criterion(logits, labels_b).item() * len(labels_b)
                preds          = leaf_indices_t[logits[:, leaf_indices_t].argmax(dim=1)]
                val_correct   += (preds == labels_b).sum().item()
                val_total     += len(labels_b)

        val_loss = val_loss_sum / val_total
        val_acc  = val_correct  / val_total
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        improved = val_acc > best_val_acc
        if improved:
            best_val_acc = val_acc
            torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                        'val_acc': val_acc, 'label': label}, save_path)

        print(f'  [{label}] Epoch {epoch}/{N_EPOCHS} | train={train_loss:.4f} | '
              f'val={val_loss:.4f} | val_acc={val_acc:.4f} | {"** BEST **" if improved else ""} | {time.time()-t_epoch:.0f}s')

    print(f'\n  [{label}] Total time : {(time.time()-t_start)/60:.1f} min  |  Best val acc : {best_val_acc:.4f}')
    return history, best_val_acc

print('[OK] Training loop defined')

## 7. Train HCE Model

In [ ]:
# v9 Stage 1: load v8 checkpoint instead of training.
# If the checkpoint is missing for any reason, fall through to a normal training pass.
hce_model = build_fresh_model()
if os.path.exists(HCE_MODEL_PATH):
    print(f'  Loading existing checkpoint: {HCE_MODEL_PATH}')
    ckpt = torch.load(HCE_MODEL_PATH, map_location=device)
    state_dict = ckpt['model_state_dict'] if isinstance(ckpt, dict) and 'model_state_dict' in ckpt else ckpt
    missing, unexpected = hce_model.load_state_dict(state_dict, strict=False)
    if missing or unexpected:
        print(f'  ⚠ state_dict mismatch — missing={len(missing)}  unexpected={len(unexpected)}')
        print(f'    (check that v9 vocab matches v8\'s; n_classes={n_classes})')
    hce_history, hce_best_acc = None, ckpt.get('best_val_acc', None) if isinstance(ckpt, dict) else None
    print('  [OK] v8 checkpoint loaded — skipping training')
else:
    print(f'  No checkpoint at {HCE_MODEL_PATH}; training from scratch')
    hce_history, hce_best_acc = train_model(hce_model, hce_criterion, HCE_MODEL_PATH, 'HCE')


## 7.5 Temperature Calibration (fit on validation)


In [ ]:
# v9 Stage 1: load v8 temperature if present, otherwise fit a fresh one.
import os
if os.path.exists(TEMPERATURE_PATH):
    T_value = float(torch.load(TEMPERATURE_PATH, map_location='cpu'))
    print(f'  Loaded existing temperature T={T_value:.3f} from {TEMPERATURE_PATH}')
else:
    print('Fitting temperature on validation set ...')
    ckpt = torch.load(HCE_MODEL_PATH, map_location=device)
    hce_model.load_state_dict(ckpt['model_state_dict'] if isinstance(ckpt, dict) and 'model_state_dict' in ckpt else ckpt)
    hce_model.eval()

    val_logits_all, val_labels_all = [], []
    with torch.no_grad():
        for batch in tqdm(val_loader, desc='  val logits', leave=False):
            logits = hce_model(batch['input_ids'].to(device), batch['attention_mask'].to(device))
            leaf_logits = logits[:, leaf_indices_t]
            val_logits_all.append(leaf_logits.detach().cpu())
            val_labels_all.append(batch['label'].cpu())
    val_logits = torch.cat(val_logits_all, dim=0)
    val_labels_t = torch.cat(val_labels_all, dim=0)
    leaf_to_local = {leaf_idx: i for i, leaf_idx in enumerate(leaf_indices)}
    val_local = torch.tensor([leaf_to_local.get(int(l), -1) for l in val_labels_t.tolist()], dtype=torch.long)
    valid_mask = val_local >= 0
    val_logits_f = val_logits[valid_mask].to(device)
    val_local_f  = val_local[valid_mask].to(device)
    print(f'  Val cells (with leaf labels) : {valid_mask.sum().item():,}  / {len(val_local):,}')

    T = torch.tensor([1.0], device=device, requires_grad=True)
    nll_loss = nn.CrossEntropyLoss()
    optimizer = torch.optim.LBFGS([T], lr=0.1, max_iter=100)
    def closure():
        optimizer.zero_grad()
        loss = nll_loss(val_logits_f / T.clamp(min=0.05), val_local_f)
        loss.backward()
        return loss
    optimizer.step(closure)
    T_value = float(T.clamp(min=0.05).item())

    def compute_ece(probs, labels, n_bins=15):
        confs, preds = probs.max(dim=1)
        accs = (preds == labels).float()
        bins = torch.linspace(0, 1, n_bins+1)
        ece = 0.0
        for k in range(n_bins):
            m = (confs > bins[k]) & (confs <= bins[k+1])
            if m.sum() > 0:
                ece += (m.float().mean() * (accs[m].mean() - confs[m].mean()).abs()).item()
        return ece
    probs_pre  = torch.softmax(val_logits_f, dim=1)
    probs_post = torch.softmax(val_logits_f / T_value, dim=1)
    print(f'  Fitted T = {T_value:.3f}')
    print(f'  ECE pre  = {compute_ece(probs_pre.cpu(),  val_local_f.cpu())*100:.2f}%')
    print(f'  ECE post = {compute_ece(probs_post.cpu(), val_local_f.cpu())*100:.2f}%')

    # Save to V9 dir (don't overwrite v8 temperature)
    v9_temp_path = os.path.join(OUT_DIR, 'temperature_v9.pt')
    torch.save(T_value, v9_temp_path)
    print(f'[OK] Saved temperature to {v9_temp_path}')


## 8. Evaluate HCE on Test Set

In [ ]:
def evaluate_model(model, ckpt_path, label):
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    print(f'  [{label}] Loaded epoch {ckpt["epoch"]}, val_acc={ckpt["val_acc"]:.4f}')
    model.eval()
    preds_all, trues_all = [], []
    with torch.no_grad():
        for batch in tqdm(test_loader, desc=f'  [{label}] Evaluating', leave=True):
            logits = model(batch['input_ids'].to(device), batch['attention_mask'].to(device))
            preds  = leaf_indices_t[logits[:, leaf_indices_t].argmax(dim=1)]
            preds_all.extend(preds.cpu().numpy())
            trues_all.extend(batch['label'].numpy())
    return np.array(preds_all), np.array(trues_all)


def compute_metrics(trues, preds, organ_id=None):
    if organ_id is not None:
        mask  = test_organ == organ_id
        trues = trues[mask]; preds = preds[mask]
    if len(trues) == 0: return None
    unique = np.unique(trues)
    acc    = accuracy_score(trues, preds)
    p, r, f, _ = precision_recall_fscore_support(trues, preds, labels=unique, average='macro', zero_division=0)
    p_pc, r_pc, f_pc, sup = precision_recall_fscore_support(trues, preds, labels=unique, zero_division=0)
    per_class = pd.DataFrame({
        'cell_type': [class_names[i] for i in unique],
        'precision': p_pc, 'recall': r_pc, 'f1': f_pc, 'support': sup,
    })
    return {
        'accuracy': acc, 'macro_recall': r, 'macro_f1': f, 'macro_precision': p,
        'zero_recall_count': int((per_class['recall'] == 0).sum()),
        'recall_50_pct': float((per_class['recall'] >= 0.5).mean()),
        'recall_80_pct': float((per_class['recall'] >= 0.8).mean()),
        'per_class': per_class,
        'n_samples': len(trues),
    }


print('Evaluating HCE model ...')
hce_preds, hce_trues = evaluate_model(hce_model, HCE_MODEL_PATH, 'HCE')

results_hce = {}
for cfg in ORGAN_CONFIGS:
    m = compute_metrics(hce_trues, hce_preds, cfg['id'])
    if m: results_hce[cfg['name']] = m
results_hce['COMBINED'] = compute_metrics(hce_trues, hce_preds)

print('\n[OK] HCE evaluation complete')

## 9. HCE Test Set Metrics

In [ ]:
print('=' * 80)
print('  HCE v9 — TEST SET METRICS')
print('=' * 80)
metric_cols = ['accuracy', 'macro_recall', 'macro_f1', 'zero_recall_count',
               'recall_50_pct', 'recall_80_pct', 'n_samples']
rows = []
for name, m in results_hce.items():
    rows.append({'organ': name, **{k: m[k] for k in metric_cols}})
df_metrics = pd.DataFrame(rows)
print(df_metrics.to_string(index=False, float_format=lambda v: f'{v:.4f}'))

## 10. Training Curves

In [ ]:
# v9 Stage 1: skip training curves when reusing v8 checkpoint (no history available)
if hce_history is None:
    print('  Skipping training curves — checkpoint loaded from v8 (no training history)')
else:
    epochs_ax = np.arange(1, N_EPOCHS + 1)
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    axes[0].plot(epochs_ax, hce_history['val_acc'], 'o-', label='HCE v9')
    axes[0].set_title('Validation Accuracy'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Val Acc')
    axes[0].set_ylim(0, 1); axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].plot(epochs_ax, hce_history['val_loss'], 'o-', label='HCE v9')
    axes[1].set_title('Validation Loss'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Val Loss')
    axes[1].legend(); axes[1].grid(alpha=0.3)
    axes[2].plot(epochs_ax, hce_history['train_loss'], 'o-', label='HCE v9')
    axes[2].set_title('Training Loss'); axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Train Loss')
    axes[2].legend(); axes[2].grid(alpha=0.3)
    plt.suptitle('HCE v9 Training Curves')
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, 'hce_v9_training_curves.png'), dpi=150, bbox_inches='tight')
    plt.show()


## 11. Per-Class Recall by Organ

In [ ]:
def plot_recall(name, m, ax):
    pc = m['per_class'].sort_values('recall', ascending=True)
    ax.barh(range(len(pc)), pc['recall'], color='steelblue', alpha=0.8)
    ax.set_yticks(range(len(pc)))
    ax.set_yticklabels(pc['cell_type'], fontsize=7)
    ax.set_xlim(0, 1.05)
    ax.axvline(0.5, color='gray', linestyle='--', alpha=0.5, linewidth=0.8)
    ax.set_xlabel('Recall')
    ax.set_title(f'{name}  (n={m["n_samples"]}, mean={m["per_class"]["recall"].mean():.3f})', fontsize=10)
    ax.grid(axis='x', alpha=0.3)

organ_names = [c['name'] for c in ORGAN_CONFIGS if c['name'] in results_hce]
fig, axes = plt.subplots(len(organ_names), 1,
                         figsize=(8, max(3, 0.35 * sum(len(results_hce[n]['per_class']) for n in organ_names))))
if len(organ_names) == 1: axes = [axes]
for ax, name in zip(axes, organ_names):
    plot_recall(name, results_hce[name], ax)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'hce_v7_per_class_recall.png'), dpi=150, bbox_inches='tight')
plt.show()

## 12. Recall Distribution

In [ ]:
hce_recalls = results_hce['COMBINED']['per_class']['recall'].values
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

axes[0].hist(hce_recalls, bins=15, alpha=0.7, color='steelblue', edgecolor='black')
axes[0].axvline(0.5, color='red', linestyle='--', alpha=0.6, label='50% threshold')
axes[0].set_title(f'Recall Distribution (n={len(hce_recalls)} classes)')
axes[0].set_xlabel('Per-class recall'); axes[0].set_ylabel('# classes')
axes[0].legend(); axes[0].grid(alpha=0.3)

sorted_r = np.sort(hce_recalls)
cdf      = np.arange(1, len(sorted_r) + 1) / len(sorted_r)
axes[1].plot(sorted_r, cdf, '-', linewidth=2, color='steelblue')
axes[1].axvline(0.5, color='gray', linestyle='--', alpha=0.5)
axes[1].set_title('CDF of per-class recall'); axes[1].set_xlabel('Recall threshold'); axes[1].set_ylabel('Fraction of classes')
axes[1].grid(alpha=0.3)

print(f'  HCE — mean recall: {hce_recalls.mean():.4f}  |  median: {np.median(hce_recalls):.4f}  |  0% classes: {(hce_recalls == 0).sum()}')

plt.suptitle('HCE v7 — Combined Test-Set Recall')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'hce_v7_recall_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

## 13. Zero-Shot Inference *(6 lab datasets)*

In [ ]:
class InferenceDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length):
        self.texts, self.tokenizer, self.max_length = texts, tokenizer, max_length
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx] if self.texts[idx].strip() else '[PAD]',
            truncation=True, max_length=self.max_length,
            padding='max_length', return_tensors='pt')
        return {'input_ids': enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0)}


# ── v9 Stage 1, S1-A: mask "(unspecified)" leaves from leaf argmax ─────────
# These leaves are deduplicate_hierarchy() artifacts and become catch-all attractors.
# Roll-up still uses the FULL leaf set so ancestor probability mass is preserved.
specific_leaf_classes = [c for c in leaf_classes if not c.endswith(' (unspecified)')]
specific_leaf_indices = [class_to_idx[c] for c in specific_leaf_classes]
specific_leaf_indices_t = torch.tensor(specific_leaf_indices, device=device)
masked_count = len(leaf_indices) - len(specific_leaf_indices)
print(f'  S1-A: masked {masked_count} "(unspecified)" leaves from argmax  ({len(specific_leaf_indices)} specific leaves remain)')
print(f'  Masked classes (first 15): {[c for c in leaf_classes if c.endswith(" (unspecified)")][:15]}')


# ── Ancestor reachability + node-depth helpers for roll-up ─────────────────
ancestors_of_leaf = {}  # leaf_idx -> list of ancestor class indices (incl. itself)
for leaf_idx in leaf_indices:
    ancestors_of_leaf[leaf_idx] = [i for i in range(n_classes) if R_np[i, leaf_idx] > 0]

def _compute_depths():
    depths = {name: None for name in class_names}
    def depth_of(name, seen=None):
        seen = seen or set()
        if name in seen: return 0
        if depths[name] is not None: return depths[name]
        parent = combined_ontology.get(name)
        if parent is None or parent not in depths:
            depths[name] = 0
        else:
            depths[name] = 1 + depth_of(parent, seen | {name})
        return depths[name]
    for nm in class_names: depth_of(nm)
    return depths
node_depth = _compute_depths()
depth_arr  = np.array([node_depth[nm] for nm in class_names], dtype=int)


# ── v9 Stage 1, S1-C: adaptive roll-up threshold ───────────────────────────
# v8 used tau_parent=0.7 fixed → T cells rolled up to "leukocyte" (too broad).
# Adaptive: tau = clip(2.5 × leaf_max_p, 0.4, 1.0). When leaf is uncertain (e.g. 0.15),
# threshold is just 0.4 — closer ancestors like "T cell" / "CD4-positive..." can qualify.
def rollup_pred(leaf_probs_row, tau_leaf=TAU_LEAF):
    """leaf_probs_row is keyed by leaf_indices (FULL leaf set, incl. unspecified)."""
    leaf_max_idx = int(np.argmax(leaf_probs_row))
    leaf_max_p   = float(leaf_probs_row[leaf_max_idx])
    if leaf_max_p >= tau_leaf:
        return leaf_indices[leaf_max_idx], leaf_max_p, False
    s = np.zeros(n_classes, dtype=float)
    for k, leaf_idx in enumerate(leaf_indices):
        p = leaf_probs_row[k]
        if p <= 0: continue
        for anc_idx in ancestors_of_leaf[leaf_idx]:
            s[anc_idx] += p
    tau_adaptive = max(0.4, min(1.0, 2.5 * leaf_max_p))
    candidate_mask = s >= tau_adaptive
    if not candidate_mask.any():
        return leaf_indices[leaf_max_idx], leaf_max_p, False
    cand_idx = np.where(candidate_mask)[0]
    # Prefer deepest ancestor (closest to leaf) — tie-break on higher accumulated mass
    best = max(cand_idx, key=lambda i: (depth_arr[i], s[i]))
    return int(best), float(s[best]), True


def run_inference(model, texts, label, temperature=1.0):
    """v9 Stage 1: leaf argmax uses specific_leaf_indices_t (S1-A masking);
    roll-up uses FULL leaf set (unchanged)."""
    inf_loader = DataLoader(
        InferenceDataset(texts, tokenizer, MAX_SEQ_LEN),
        batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=2, pin_memory=True
    )
    leaf_preds, leaf_confs = [], []
    lineage_preds, lineage_confs, lineage_rolled = [], [], []
    model.eval()
    with torch.no_grad():
        for batch in tqdm(inf_loader, desc=f'    [{label}] infer', leave=False):
            logits = model(batch['input_ids'].to(device), batch['attention_mask'].to(device))
            # Leaf argmax: restricted to specific (non-unspecified) leaves
            spec_logits = logits[:, specific_leaf_indices_t]
            spec_probs = torch.softmax(spec_logits / temperature, dim=1)
            pred_pos = spec_probs.argmax(dim=1)
            leaf_preds.extend(specific_leaf_indices_t[pred_pos].cpu().numpy())
            leaf_confs.extend(spec_probs.max(dim=1).values.cpu().numpy())
            # Roll-up: full leaf set (preserves ancestor accumulation including via unspecified)
            full_logits = logits[:, leaf_indices_t]
            full_probs = torch.softmax(full_logits / temperature, dim=1).cpu().numpy()
            for row in full_probs:
                idx, p, rolled = rollup_pred(row)
                lineage_preds.append(idx)
                lineage_confs.append(p)
                lineage_rolled.append(rolled)
    leaf_names    = [class_names[i] for i in leaf_preds]
    lineage_names = [class_names[i] for i in lineage_preds]
    return (leaf_names, np.array(leaf_confs),
            lineage_names, np.array(lineage_confs), np.array(lineage_rolled))


# ── Load temperature ────────────────────────────────────────────────────────
try:
    TEMPERATURE = float(torch.load(TEMPERATURE_PATH, map_location='cpu'))
    print(f'  Loaded temperature T={TEMPERATURE:.3f}')
except (FileNotFoundError, OSError):
    TEMPERATURE = 1.0
    print('  No fitted temperature found — using T=1.0')


# ── v9 Stage 1, S1-B: REAL Brain_normal protocol diagnostic ────────────────
# Inspect adata.obs columns and median total counts on a 200-cell sample.
# snRNA-seq cells typically have median total_counts 1k–8k; scRNA-seq 8k–50k+.
print('\n  S1-B: protocol diagnostic for Brain_normal / All_cells (glioma):')
import random
random.seed(SEED)
for lab_cfg in LAB_CONFIGS:
    if lab_cfg['name'] not in ('Brain_normal', 'All_cells (glioma)'):
        continue
    try:
        ad = sc.read_h5ad(lab_cfg['path'], backed='r')
        cols = list(ad.obs.columns)
        sample_idx = random.sample(range(ad.n_obs), min(200, ad.n_obs))
        sample_idx_sorted = sorted(sample_idx)  # h5 needs monotonic for some readers
        if sp.issparse(ad.X):
            sub = ad.X[sample_idx_sorted]
            sample_totals = np.asarray(sub.sum(axis=1)).flatten()
        else:
            sample_totals = np.asarray(ad.X[sample_idx_sorted]).sum(axis=1).flatten()
        median_total = float(np.median(sample_totals))
        guess = 'snRNA-seq' if median_total < 8000 else 'scRNA-seq'
        print(f'    [{lab_cfg["name"]}]  obs cols (first 10): {cols[:10]}')
        print(f'      median total counts on 200-cell sample = {median_total:.0f}  →  likely {guess}')
        print(f'      current snrna_seq flag in LAB_CONFIGS = {lab_cfg["snrna_seq"]}')
        if (guess == 'snRNA-seq') != lab_cfg['snrna_seq']:
            print(f'      ⚠ MISMATCH — consider flipping snrna_seq for {lab_cfg["name"]}')
    except Exception as e:
        print(f'    [{lab_cfg["name"]}] diagnostic failed: {e}')


lab_results_hce = {}

for lab_cfg in LAB_CONFIGS:
    lab_name  = lab_cfg['name']
    label_col = lab_cfg['label_col']
    sn_flag   = lab_cfg.get('snrna_seq', False)
    print(f'\n  --- {lab_name} ---  (snrna_seq={sn_flag})')

    adata_lab    = sc.read_h5ad(lab_cfg['path'])
    lab_gene_sym = get_gene_symbols(adata_lab)
    lab_gene_keep = keep_gene_mask(lab_gene_sym)
    lab_gene_lengths = get_gene_lengths(adata_lab, lab_gene_sym) if sn_flag else None
    true_labels  = adata_lab.obs[label_col].astype(str).values
    X_lab        = adata_lab.X
    texts = [cell_to_text_dense(X_lab[i], lab_gene_sym, TOP_K_GENES,
                                 gene_keep=lab_gene_keep,
                                 gene_lengths=lab_gene_lengths,
                                 length_normalize=sn_flag)
             for i in tqdm(range(adata_lab.n_obs), desc='    text', leave=False)]

    leaf_names, leaf_confs, lineage_names, lineage_confs, lineage_rolled = run_inference(
        hce_model, texts, 'HCE', temperature=TEMPERATURE
    )

    lab_results_hce[lab_name] = pd.DataFrame({
        'true_label': true_labels,
        'pred_leaf': leaf_names,
        'conf_leaf': leaf_confs,
        'pred_lineage': lineage_names,
        'conf_lineage': lineage_confs,
        'rolled_up': lineage_rolled,
    })

    print(f'  Roll-up rate: {lineage_rolled.mean()*100:.1f}%')
    print(f'  {"True label":<40} {"HCE leaf":>40} {"HCE lineage":>40}')
    print('  ' + '-' * 122)
    df = lab_results_hce[lab_name]
    for true_lbl in sorted(np.unique(true_labels)):
        sub = df[df['true_label']==true_lbl]
        leaf_top = sub['pred_leaf'].value_counts()
        lin_top  = sub['pred_lineage'].value_counts()
        l_s = f'{leaf_top.index[0]}  ({leaf_top.iloc[0]/leaf_top.sum()*100:.0f}%)' if len(leaf_top) else 'N/A'
        n_s = f'{lin_top.index[0]}  ({lin_top.iloc[0]/lin_top.sum()*100:.0f}%)' if len(lin_top) else 'N/A'
        print(f'  {true_lbl:<40} {l_s:>40} {n_s:>40}')

print('\n[OK] Zero-shot inference complete  (S1-A masking + S1-C adaptive roll-up active)')


## 14. Deeper Biological-Accuracy Analysis

Top-5 predictions per true class, lineage-match rate, and ancestor recall on the in-distribution test set.

In [ ]:
# Coarse lineage assignment for any free-form label
LINEAGE_RULES = [
    ('Myeloid',                 ['macroph','monocyte','microglia','dendritic',' dc ','tam-','tam_','tam ',
                                 'kupffer','neutrophil','mast cell','eosinophil','basophil','myeloid',
                                 'granulocyte','histiocyt','langerhans']),
    ('Lymphoid',                ['t cell','t_cell','cd4','cd8','b cell','b_cell','nk cell','natural killer',
                                 'plasma cell','plasmablast','lymphocyte','lymphoid','innate lymphoid',
                                 ' ilc','follicular helper','regulatory t','gamma-delta','germinal center']),
    ('Erythroid/Megakaryocyte', ['erythro','reticulocyt','megakaryo','platelet']),
    ('Hematopoietic Progenitor',['hematopoietic','hsc',' progenitor','precursor cell']),
    ('Endothelial',             ['endothel','high endothelial venule','ec arterial','ec venous',
                                 'ec general','lymphatic ec','arterial_endothel','venous_endothel',
                                 'capillary_endothel']),
    ('Mesenchymal/Stromal',     ['fibroblast','stellate','pericyte','smooth muscle','smooth_muscle',
                                 'adipocyte','mesenchym','perivascular','meningeal','stroma',
                                 'myofibro','tenocyte','chondrocyte']),
    ('Epithelial',              ['hepatocyt','hepatoblast','cholangio','epithel','enterocyt',
                                 'alveolar type','hepatic cell','goblet','keratino','urothel','tuft']),
    ('Glial',                   ['astrocyt','oligodendro','opc','glia','bergmann','ependymal',
                                 'choroid plexus']),
    ('Neuronal',                ['neuron','interneuron','intratelencephal','hippocampal','thalamic',
                                 'amygdala','rhombic','mammillary','medium spiny',' npc','purkinje',
                                 'granule cell','msn','gabaergic']),
    ('Tumor (glioma-like)',     ['ac-like','mes-like','npc-like','opc-like']),
]
def assign_lineage(label):
    if label is None or (isinstance(label, float) and pd.isna(label)): return 'Other'
    s = ' ' + str(label).lower().strip() + ' '
    for lineage, kws in LINEAGE_RULES:
        for kw in kws:
            if kw in s: return lineage
    return 'Other'

# (1) Top-5 predictions per true class
print('=' * 100)
print('  (1) TOP-5 PREDICTIONS PER TRUE CLASS (zero-shot)')
print('=' * 100)
for lab_name in lab_results_hce.keys():
    print(f'\n--- {lab_name} ---')
    df = lab_results_hce[lab_name]
    for true_lbl in sorted(df['true_label'].unique()):
        n = int((df['true_label'] == true_lbl).sum())
        top = df[df['true_label']==true_lbl]['pred_leaf'].value_counts(normalize=True).head(5)
        print(f'\n  {true_lbl}  (n={n})')
        print(f'    HCE : ' + '  |  '.join([f'{p}: {v*100:.0f}%' for p, v in top.items()]))

# (2) Lineage-match rate per dataset
print('\n' + '=' * 100)
print('  (2) LINEAGE-MATCH RATE')
print('=' * 100)
print(f'\n  {"Dataset":<26}{"HCE":>10}')
print('  ' + '-' * 36)
for lab_name in lab_results_hce.keys():
    df = lab_results_hce[lab_name]
    rate = (df['pred_leaf'].map(assign_lineage) == df['true_label'].map(assign_lineage)).mean()
    print(f'  {lab_name:<26}{rate*100:>9.1f}%')

# Combined pooled lineage match
combined = pd.concat(list(lab_results_hce.values()), ignore_index=True)
pooled   = (combined['pred_leaf'].map(assign_lineage) == combined['true_label'].map(assign_lineage)).mean()
print(f'\n  POOLED across {len(lab_results_hce)} datasets : {pooled*100:.1f}%')

# (3) Ancestor recall @ k on in-distribution test set
def ancestor_set(node, ontology, max_depth=30):
    out, cur, depth = set(), ontology.get(node), 0
    while cur is not None and depth < max_depth and cur not in out:
        out.add(cur); cur = ontology.get(cur); depth += 1
    return out

def lca_hops(true_name, pred_name, ontology, max_k=30):
    if true_name == pred_name: return 0
    true_or_anc = {true_name} | ancestor_set(true_name, ontology, max_k)
    cur, k = pred_name, 0
    while cur is not None and k < max_k:
        if cur in true_or_anc: return k
        cur = ontology.get(cur); k += 1
    return None

def ancestor_recall_table(preds, trues):
    cache, depths = {}, []
    for t, p in zip(trues, preds):
        key = (int(t), int(p))
        if key not in cache:
            cache[key] = lca_hops(class_names[t], class_names[p], combined_ontology)
        depths.append(cache[key])
    arr = np.array([d if d is not None else 999 for d in depths])
    return {
        'leaf-exact (k=0)':  float((arr == 0).mean()),
        'within k<=1':       float((arr <= 1).mean()),
        'within k<=2':       float((arr <= 2).mean()),
        'within k<=3':       float((arr <= 3).mean()),
        'any shared anc':    float((arr <= 30).mean()),
    }

print('\n' + '=' * 100)
print('  (3) ANCESTOR RECALL @ k  (in-distribution test set)')
print('=' * 100)
tbl = ancestor_recall_table(hce_preds, hce_trues)
print(f'\n  {"Metric":<22}{"HCE":>10}')
print('  ' + '-' * 32)
for k, v in tbl.items():
    print(f'  {k:<22}{v*100:>9.2f}%')

print('\n  Per-organ ancestor-recall @ k<=2:')
print(f'  {"Organ":<20}{"HCE":>10}')
print('  ' + '-' * 30)
test_organ_np = np.asarray(test_organ)
for cfg in ORGAN_CONFIGS:
    mask = (test_organ_np == cfg['id'])
    if mask.sum() == 0: continue
    sub = ancestor_recall_table(hce_preds[mask], hce_trues[mask])
    print(f'  {cfg["name"]:<20}{sub["within k<=2"]*100:>9.2f}%')

print('\n[OK] Deep biological-accuracy analysis complete')

## 15. Confusion Matrix Visualizations

In [ ]:
import seaborn as sns

LINEAGE_ORDER = ['Myeloid','Lymphoid','Erythroid/Megakaryocyte','Hematopoietic Progenitor',
                 'Endothelial','Mesenchymal/Stromal','Epithelial','Glial','Neuronal',
                 'Tumor (glioma-like)','Other']

def lineage_confusion(df, order=LINEAGE_ORDER):
    tl = df['true_label'].map(assign_lineage)
    pl = df['pred_leaf'].map(assign_lineage)
    cm = pd.crosstab(tl, pl, normalize='index') * 100
    rows = [o for o in order if o in cm.index]
    cols = [o for o in order if o in cm.columns]
    return cm.reindex(index=rows, columns=cols).fillna(0)


# Per-dataset lineage heatmaps (HCE only)
n_ds = len(LAB_CONFIGS)
fig, axes = plt.subplots(n_ds, 1, figsize=(10, 4.5 * n_ds))
if n_ds == 1: axes = [axes]
fig.suptitle('HCE v7 Zero-Shot Lineage Confusion', fontsize=14, y=1.00)
for ax, lab_cfg in zip(axes, LAB_CONFIGS):
    name = lab_cfg['name']
    df = lab_results_hce[name]
    true_lin = df['true_label'].map(assign_lineage)
    acc      = (df['pred_leaf'].map(assign_lineage) == true_lin).mean() * 100
    cm = lineage_confusion(df)
    sns.heatmap(cm, annot=True, fmt='.0f', cmap='Blues', vmin=0, vmax=100, cbar=False,
                ax=ax, annot_kws={'size': 9}, linewidths=0.4, linecolor='lightgray')
    ax.set_title(f'{name}  (lineage match = {acc:.1f}%)', fontsize=11)
    ax.set_xlabel('Predicted lineage'); ax.set_ylabel('True lineage')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'hce_v7_cm_lineage_per_dataset.png'), dpi=150, bbox_inches='tight')
plt.show()


# Class-level heatmaps per dataset
TOP_N_PREDS = 15
for lab_cfg in LAB_CONFIGS:
    name = lab_cfg['name']
    df = lab_results_hce[name]
    cm = pd.crosstab(df['true_label'], df['pred_leaf'], normalize='index') * 100
    top_cols = cm.sum(axis=0).sort_values(ascending=False).head(TOP_N_PREDS).index.tolist()
    cm = cm.reindex(columns=top_cols, fill_value=0)
    h = max(4, 0.45 * len(cm.index) + 1.5)
    w = max(8, 0.55 * len(cm.columns) + 2.0)
    fig, ax = plt.subplots(figsize=(w, h))
    sns.heatmap(cm, annot=True, fmt='.0f', cmap='Blues', vmin=0, vmax=100,
                cbar_kws={'label': '% of true class'},
                ax=ax, annot_kws={'size': 7}, linewidths=0.3, linecolor='whitesmoke')
    ax.set_title(f'{name} — HCE class-level confusion (top {len(top_cols)} predicted classes)', fontsize=12)
    ax.set_xlabel('Predicted cell type'); ax.set_ylabel('True label')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=7)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8)
    plt.tight_layout()
    safe = name.replace(' ','_').replace('(','').replace(')','').replace('/','_')
    plt.savefig(os.path.join(OUT_DIR, f'hce_v7_cm_classlevel_{safe}.png'), dpi=150, bbox_inches='tight')
    plt.show()


# Combined pooled lineage matrix
combined_hce = pd.concat(list(lab_results_hce.values()), ignore_index=True)
true_lin_all = combined_hce['true_label'].map(assign_lineage)
acc_all      = (combined_hce['pred_leaf'].map(assign_lineage) == true_lin_all).mean() * 100

fig, ax = plt.subplots(figsize=(11, 8))
cm = lineage_confusion(combined_hce)
sns.heatmap(cm, annot=True, fmt='.0f', cmap='Blues', vmin=0, vmax=100,
            cbar_kws={'label': '% of true row'},
            ax=ax, annot_kws={'size': 10}, linewidths=0.5, linecolor='lightgray')
ax.set_title(f'HCE v7 — Combined Zero-Shot Lineage Confusion (all {len(LAB_CONFIGS)} lab datasets pooled)\noverall lineage match = {acc_all:.1f}%', fontsize=12)
ax.set_xlabel('Predicted lineage'); ax.set_ylabel('True lineage')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'hce_v7_cm_lineage_combined.png'), dpi=150, bbox_inches='tight')
plt.show()

print('\n[OK] Confusion matrix visualizations complete')

## 16. Final Summary

In [ ]:
print('=' * 80)
print('  HCE v9 — FINAL SUMMARY')
print('=' * 80)

print('\nTRAINING')
print(f'  Best validation accuracy : {hce_best_acc:.4f}')

print('\nTEST SET (in-distribution)')
m = results_hce['COMBINED']
print(f'  accuracy            : {m["accuracy"]:.4f}')
print(f'  macro_recall        : {m["macro_recall"]:.4f}')
print(f'  macro_f1            : {m["macro_f1"]:.4f}')
print(f'  zero_recall_count   : {m["zero_recall_count"]}')
print(f'  recall_50_pct       : {m["recall_50_pct"]:.4f}')
print(f'  recall_80_pct       : {m["recall_80_pct"]:.4f}')

print('\nPER-ORGAN MACRO RECALL (test set)')
print(f'  {"Organ":<20}{"HCE":>10}')
print('  ' + '-' * 30)
for cfg in ORGAN_CONFIGS:
    if cfg['name'] in results_hce:
        r = results_hce[cfg['name']]['macro_recall']
        print(f'  {cfg["name"]:<20}{r:>10.4f}')

print('\nZERO-SHOT LINEAGE MATCH (per lab dataset)')
print(f'  {"Dataset":<26}{"HCE":>10}')
print('  ' + '-' * 36)
for lab_name in lab_results_hce.keys():
    df = lab_results_hce[lab_name]
    rate = (df['pred_leaf'].map(assign_lineage) == df['true_label'].map(assign_lineage)).mean()
    print(f'  {lab_name:<26}{rate*100:>9.1f}%')
print(f'  {"POOLED":<26}{acc_all:>9.1f}%')

print('\n  Outputs saved to:', OUT_DIR)

## 17. v6 / v7 / v8 / v9 Comparison


In [ ]:
# v9 Stage 1 comparison: in-distribution test metrics come from the v8 checkpoint
# (reused, no retrain) so v9-test ≡ v8-test. Zero-shot lineage-match below reflects
# Stage 1 changes (S1-A masking + S1-C adaptive roll-up + Brain_normal flag flip).
import os, pandas as pd

def _try_read_csv(path):
    return pd.read_csv(path) if os.path.exists(path) else None

v6_csv = _try_read_csv('multi_tissue_v6_comparison_results/hce_vs_ce_comparison.csv')
v7_csv = _try_read_csv('multi_tissue_v7_hce_only_results/hce_v7_test_metrics.csv')
v8_csv = _try_read_csv('multi_tissue_v8_results/hce_v8_test_metrics.csv')

rows = []
def add(row, organ_name):
    if v6_csv is not None:
        r6 = v6_csv[v6_csv['organ'] == organ_name]
        if len(r6):
            row['v6_acc']      = float(r6.iloc[0]['HCE_accuracy'])
            row['v6_macro_f1'] = float(r6.iloc[0]['HCE_macro_f1'])
    if v7_csv is not None:
        r7 = v7_csv[v7_csv['organ'] == organ_name]
        if len(r7):
            row['v7_acc']      = float(r7.iloc[0].get('accuracy', float('nan')))
            row['v7_macro_f1'] = float(r7.iloc[0].get('macro_f1', float('nan')))
    if v8_csv is not None:
        r8 = v8_csv[v8_csv['organ'] == organ_name]
        if len(r8):
            row['v8_acc']      = float(r8.iloc[0].get('accuracy', float('nan')))
            row['v8_macro_f1'] = float(r8.iloc[0].get('macro_f1', float('nan')))

for organ_name, m in results_hce.items():
    if organ_name == 'COMBINED': continue
    row = {'organ': organ_name,
           'v9_acc': m['accuracy'], 'v9_macro_f1': m.get('macro_f1', float('nan'))}
    add(row, organ_name)
    rows.append(row)

m = results_hce.get('COMBINED', {})
combined = {'organ': 'COMBINED',
            'v9_acc': m.get('accuracy', float('nan')),
            'v9_macro_f1': m.get('macro_f1', float('nan'))}
add(combined, 'COMBINED')
rows.append(combined)

cmp_df = pd.DataFrame(rows)
cmp_df.to_csv(os.path.join(OUT_DIR, 'hce_v6_v7_v8_v9_comparison.csv'), index=False)
print('In-distribution test (note: v9 Stage 1 reuses v8 checkpoint → v9 ≡ v8 on test):')
print(cmp_df.to_string(index=False))

# Zero-shot lineage-match — differs from v8 because of S1-A masking + S1-C rollup
print('\n  v9 Stage 1 zero-shot lineage-match  (compare to v7=61.0%, v8=60.6% pooled):')
total_leaf, total_lin, total_n = 0, 0, 0
for lab_name, df in lab_results_hce.items():
    n = len(df)
    match_leaf    = sum(assign_lineage(t) == assign_lineage(p) for t,p in zip(df['true_label'], df['pred_leaf']))
    match_lineage = sum(assign_lineage(t) == assign_lineage(p) for t,p in zip(df['true_label'], df['pred_lineage']))
    delta = (match_lineage - match_leaf) / n * 100
    print(f'  {lab_name:<25}  leaf={match_leaf/n*100:5.1f}%   lineage={match_lineage/n*100:5.1f}%   Δ={delta:+5.1f}pt  (n={n})')
    total_leaf += match_leaf; total_lin += match_lineage; total_n += n
if total_n:
    print(f'  {"POOLED":<25}  leaf={total_leaf/total_n*100:5.1f}%   lineage={total_lin/total_n*100:5.1f}%   (n={total_n})')


## 18. Biology Spot-Checks (v8 acceptance criteria)


In [ ]:
# Biology spot-checks: explicit pass/fail for the regression cases identified in v7 analysis.
# Each check looks at the most-frequent prediction for a given (lab_dataset, true_label) pair
# and asserts a substring (case-insensitive) appears in either top-1 leaf or top-1 lineage.

SPOT_CHECKS = [
    # (dataset, true_label_substring, allowed_substrings_in_prediction, axis)
    ('lymphoid',          'CD4_T_cell',     ['CD4'],                           'lineage'),
    ('lymphoid',          'Treg_cell',      ['CD4', 'regulatory'],             'lineage'),
    ('lymphoid',          'CD8_T_cell',     ['CD8'],                           'leaf'),
    ('myeloid',           'monocyte_',      ['monocyte'],                      'leaf'),
    ('Brain_normal',      'oligodendrocyte', ['oligodendrocyte'],              'leaf'),
    ('Liver_normal',      'hepatocyte',     ['hepatocyte'],                    'leaf'),
    ('Liver_normal',      'cholangiocyte',  ['cholangiocyte'],                 'leaf'),
    ('Lymph_node_normal', 'LN_stroma_cell', ['fibroblast', 'stroma'],          'lineage'),
    ('Brain_normal',      'GABAergic',      ['interneuron', 'inhibitory neuron','GABA'], 'lineage'),
]

print(f'  {"Dataset":<22} {"True label match":<28} {"Top pred":<45} {"Axis":<8} {"PASS?":<6}')
print('  ' + '-' * 110)
n_pass, n_total = 0, 0
for ds_name, true_sub, allowed_subs, axis in SPOT_CHECKS:
    df = lab_results_hce.get(ds_name)
    if df is None:
        print(f'  {ds_name:<22}  (dataset not found, skipped)')
        continue
    pred_col = 'pred_lineage' if axis == 'lineage' else 'pred_leaf'
    sub = df[df['true_label'].str.contains(true_sub, case=False, na=False, regex=False)]
    if len(sub) == 0:
        print(f'  {ds_name:<22} {true_sub:<28}  (no matching cells)')
        continue
    top = sub[pred_col].value_counts()
    top_pred = top.index[0]
    passed = any(a.lower() in top_pred.lower() for a in allowed_subs)
    n_total += 1
    n_pass += int(passed)
    print(f'  {ds_name:<22} {true_sub:<28} {top_pred:<45} {axis:<8} {"PASS" if passed else "FAIL":<6}')

print(f'\n  Spot checks passed: {n_pass}/{n_total}')
print(f'  v8 acceptance target: ≥ 4 of these passing that didn\'t in v7')
